In [16]:
import warnings
from elasticsearch.exceptions import ElasticsearchWarning
import json
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
from tqdm import tqdm

In [18]:
client = Elasticsearch("http://localhost:9200")

# To hide messages for HTTPS
warnings.simplefilter('ignore', ElasticsearchWarning)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")

# Connection to Elasticsearch 8.x
client = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic", "lcruhVaFiAjkVXltXOV2"),
    verify_certs=False
)

# Confirm connection
print(client.info())

C:\Users\monoh\anaconda3\Lib\site-packages\elasticsearch\_sync\client\__init__.py:329: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


{'name': 'LAPTOP-R9IG9QUF', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'zdYT7L05SQyrj-ptW1rzvA', 'version': {'number': '9.5.6', 'build_flavor': 'default', 'build_type': 'zip', 'build_hash': 'fd1e8011342a0988e432b999cea4bf884ca120ed', 'build_date': '2026-10-28T22:37:54.387301573Z', 'build_snapshot': False, 'lucene_version': '10.5.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [20]:
all_documents = []

file_path = "IR2026/documents.jsonl"

# Open jsonl file for reading
with open(file_path, "r", encoding="utf-8") as file:
    for line in file:
        # Convert each line from JSON string to a Python dictionary
        json_data = json.loads(line)
        all_documents.append(json_data)

print(f"{len(all_documents)} documents Successfully read!")

if len(all_documents) > 0:
    print("\nFirst document's structure:")
    print(all_documents[0])

40221 documents Successfully read!

First document's structure:
{'_id': '11547', 'title': 'Is there a word meaning "an unwanted eponym"?', 'text': 'An eponym is one way to eternal (if posthumous) fame. But is there a word meaning an eponym someone would sooner not have? (One would presume that Captain Charles _Boycott_ , Mr Justice _Lynch_ , and Patrick _Hooligan_ would not appreciate their undying notoriety.)'}


In [22]:
index_name = "ir2026_index"

# Delete the index if it already exists
if client.indices.exists(index=index_name):
    client.indices.delete(index=index_name)

# Define Stucture (Mapping & Settings)
mapping = {
    "settings": {
        "number_of_shards": 1,
        "index": {
            "similarity": {
                "default": {
                    "type": "BM25",
                    "k1": 1.5,
                    "b": 0.75
                }
            }
        },
        "analysis": {
            "analyzer": {
                "default": {
                    "type": "english"
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "title": {"type": "text"},
            "text": {"type": "text"}
        }
    }
}

# Create Index 
client.indices.create(index=index_name, body=mapping)
print(f"The index '{index_name}' was created.")

# Prepare data for Bulk Insert
bulk_documents = []
for doc in all_documents:
    bulk_documents.append({
        "_index": index_name,
        "_id": doc["_id"],  #Keep the initial ID to match with the qrels
        "_source": {
            "title": doc["title"],
            "text": doc["text"]
        }
    })


# Bulk Insert Of the documents to ElasticSearch
print("Data Insert is starting. Please wait...")
bulk(client, bulk_documents, refresh=True)

# Confirmation
count = client.cat.count(index=index_name, format="json")[0]["count"]
print(f"The Insert completed! The index has {count} documents.")

The index 'ir2026_index' was created.
Data Insert is starting. Please wait...
The Insert completed! The index has 40221 documents.


In [23]:
# Load Queries
queries = []
with open("IR2026/queries.jsonl", "r", encoding="utf-8") as file:
    for line in file:
        queries.append(json.loads(line))
        
print(f"Loaded {len(queries)} queries. Starting search...")

#Execute the search for k = 20, 30, 50

for k in tqdm([20, 30, 50]):
    results = {}
    for q in queries:
        q_id = str(q["_id"])
        q_text = q["text"]
        
        response = client.search(
            index=index_name, 
            size=k,
            query={
                "multi_match": {
                    "query": q_text,
                    "fields": ["title", "text"]
                }}
        )
        
        #Export the document IDs and BM25 score 
        returned_docs = []
        for hit in response["hits"]["hits"]:
            returned_docs.append({
                "id": hit["_id"], 
                "score": hit["_score"]
            })
            
        results[q_id] = returned_docs
        
    #Save the results to a JSON file
    filename = f"results_k{k}.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)
    print(f"Successfully saved results for k={k} in {filename}")

Loaded 1570 queries. Starting search...


 33%|████████████████████████████                                                        | 1/3 [00:25<00:50, 25.46s/it]

Successfully saved results for k=20 in results_k20.json


 67%|████████████████████████████████████████████████████████                            | 2/3 [00:43<00:21, 21.14s/it]

Successfully saved results for k=30 in results_k30.json


100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [01:06<00:00, 22.05s/it]

Successfully saved results for k=50 in results_k50.json


In [26]:
for k in [20, 30, 50]:
    input_json = f"results_k{k}.json"
    output_trec = f"run_k{k}.txt"
    try:
        with open(input_json, "r", encoding="utf-8") as f:
            results = json.load(f)    
        # Create the TREC format file for the specific k
        with open(output_trec, "w", encoding="utf-8") as out_f:
            for q_id, doc_list in results.items():
                # enumerate() gives us the rank of the document, starting from 1
                for rank, doc in enumerate(doc_list, start=1):
                    # get the ID and the BM25 score
                    doc_id = doc["id"]
                    score = doc["score"]
                    
                    trec_line = f"{q_id} Q0 {doc_id} {rank} {score:.4f} run_{k}\n"
                    out_f.write(trec_line)
                    
        print(f"Success: Created evaluation file {output_trec}")
    except FileNotFoundError:
        print(f"Error: File {input_json} not found. Ensure it was created in the previous step.")

Success: Created evaluation file run_k20.txt
Success: Created evaluation file run_k30.txt
Success: Created evaluation file run_k50.txt
